# GSE44076: GPL13667 Probe Annotation

The paired comparison is currently ranked by Affymetrix probe-set identifier. Probe annotation is required before any gene-level discussion because mappings may be missing, many probes may map to one gene, and some probe sets may contain multiple assignments.

This notebook joins GPL13667 platform annotation to the paired probe ranking. It does not collapse probes to genes, perform pathway analysis, identify biomarkers, or make biological claims.

## Setup

In [1]:
from pathlib import Path
import gzip
import shutil
import sys
from urllib.error import URLError
from urllib.request import urlopen

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis import (  # noqa: E402
    build_paired_sample_table,
    derive_pair_ids,
    paired_ttest_by_probe,
)
from src.annotation import (  # noqa: E402
    annotate_probe_ranking,
    load_gpl_annotation,
    standardize_annotation_columns,
)
from src.geo_loader import (  # noqa: E402
    align_expression_with_metadata,
    build_sample_metadata_table,
    convert_expression_to_numeric,
    load_geo_expression_table,
    read_geo_series_lines,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "GSE44076_series_matrix.txt.gz"
METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "sample_metadata.csv"
RANKING_PATH = PROJECT_ROOT / "data" / "processed" / "tumor_vs_paired_normal_paired_probe_ranking.csv"
ANNOTATION_PATH = PROJECT_ROOT / "data" / "raw" / "GPL13667.annot.gz"
ANNOTATED_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "tumor_vs_paired_normal_paired_probe_ranking_annotated.csv"

FTP_ANNOTATION_URL = "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13667/annot/GPL13667.annot.gz"
NCBI_PLATFORM_TEXT_URL = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL13667&targ=self&form=text&view=full"


## Load or Rebuild the Paired Ranking

The local paired ranking is used when available. If absent, it is rebuilt from the local series matrix and the 98 matched individual IDs established in the previous notebook.

In [2]:
if RANKING_PATH.is_file():
    paired_ranking = pd.read_csv(RANKING_PATH)
    ranking_source = RANKING_PATH.relative_to(PROJECT_ROOT)
else:
    if not RAW_PATH.is_file():
        raise FileNotFoundError(
            "The paired ranking and raw GSE44076 series matrix are both absent. "
            f"Place the series matrix at {RAW_PATH}."
        )

    expression = load_geo_expression_table(RAW_PATH)
    if METADATA_PATH.is_file():
        sample_metadata = pd.read_csv(METADATA_PATH)
    else:
        sample_metadata = build_sample_metadata_table(read_geo_series_lines(RAW_PATH))
        group_map = {
            "Healthy colon mucosa cells": "healthy_mucosa",
            "Normal distant colon mucosa cells": "paired_normal_mucosa",
            "Primary colon adenocarcinoma cells": "tumor",
        }
        sample_metadata["group"] = sample_metadata["source_name"].map(group_map)

    expression, sample_metadata = align_expression_with_metadata(expression, sample_metadata)
    expression = convert_expression_to_numeric(expression)
    paired_metadata = derive_pair_ids(sample_metadata)
    paired_metadata = paired_metadata[
        paired_metadata["group"].isin(["tumor", "paired_normal_mucosa"])
    ]
    pair_table = build_paired_sample_table(paired_metadata, "group", "pair_id")
    paired_ranking = paired_ttest_by_probe(
        expression,
        pair_table,
        tumor_column="tumor",
        normal_column="paired_normal_mucosa",
    )
    RANKING_PATH.parent.mkdir(parents=True, exist_ok=True)
    paired_ranking.to_csv(RANKING_PATH, index=False)
    ranking_source = "rebuilt from local series matrix"

print(f"Ranking source: {ranking_source}")
print(f"Ranked probes: {len(paired_ranking):,}")

Ranking source: data\processed\tumor_vs_paired_normal_paired_probe_ranking.csv
Ranked probes: 49,386


## Obtain or Load GPL13667 Annotation

A dedicated `GPL13667.annot.gz` file is not currently published in the platform's GEO FTP `annot/` directory. The local file therefore uses the full GPL13667 platform text supplied by NCBI, which contains the same platform annotation table between `!platform_table_begin` and `!platform_table_end`.

If the local file is absent, the code first attempts the conventional FTP annotation path and then falls back to NCBI's official full-text platform endpoint. Both raw sources remain excluded from Git.

In [3]:
def download_annotation(destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(FTP_ANNOTATION_URL, timeout=60) as response, destination.open("wb") as output:
            shutil.copyfileobj(response, output)
        return FTP_ANNOTATION_URL
    except (URLError, TimeoutError):
        if destination.exists():
            destination.unlink()

    try:
        with urlopen(NCBI_PLATFORM_TEXT_URL, timeout=180) as response, gzip.open(destination, "wb") as output:
            shutil.copyfileobj(response, output)
        return NCBI_PLATFORM_TEXT_URL
    except (URLError, TimeoutError) as error:
        if destination.exists():
            destination.unlink()
        raise RuntimeError(
            "GPL13667 annotation download failed. Manually download the full GPL13667 "
            "platform text from NCBI GEO and save it as data/raw/GPL13667.annot.gz."
        ) from error


if ANNOTATION_PATH.is_file():
    annotation_source = "existing local GPL13667 annotation file"
else:
    annotation_source = download_annotation(ANNOTATION_PATH)

print(f"Annotation source: {annotation_source}")
print(f"Local annotation path: {ANNOTATION_PATH.relative_to(PROJECT_ROOT)}")
print(f"Compressed size: {ANNOTATION_PATH.stat().st_size / 1_000_000:.1f} MB")

Annotation source: existing local GPL13667 annotation file
Local annotation path: data\raw\GPL13667.annot.gz
Compressed size: 23.0 MB


## Parse and Standardize the Platform Table

The parser tolerates common alternative column names and standardizes the fields to `probe_id`, `gene_symbol`, `gene_title`, and `entrez_id`. GEO missing-value markers such as `---` remain missing.

In [4]:
raw_annotation = load_gpl_annotation(ANNOTATION_PATH)
annotation = standardize_annotation_columns(raw_annotation)

print(f"Platform table shape: {raw_annotation.shape}")
print(f"Standardized annotation shape: {annotation.shape}")
print(f"Duplicate probe IDs: {annotation['probe_id'].duplicated().sum()}")
display(annotation.head())

Platform table shape: (49386, 43)
Standardized annotation shape: (49386, 4)
Duplicate probe IDs: 0


,probe_id,gene_symbol,gene_title,entrez_id
0,11715100_at,HIST1H3G,"histone cluster 1, H3g",8355
1,11715101_s_at,HIST1H3G,"histone cluster 1, H3g",8355
2,11715102_x_at,HIST1H3G,"histone cluster 1, H3g",8355
3,11715103_x_at,TNFAIP8L1,"tumor necrosis factor, alpha-induced protein 8...",126282
4,11715104_s_at,OTOP2,otopetrin 2,92736


## Annotate the Paired Probe Ranking

A left join preserves every ranked probe, including rows without a gene symbol.

In [5]:
annotated_ranking = annotate_probe_ranking(paired_ranking, raw_annotation)
ANNOTATED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
annotated_ranking.to_csv(ANNOTATED_OUTPUT_PATH, index=False)

probes_with_symbol = int(annotated_ranking["gene_symbol"].notna().sum())
probes_without_symbol = int(annotated_ranking["gene_symbol"].isna().sum())

print(f"Probes in ranking: {len(annotated_ranking):,}")
print(f"Probes with gene symbol: {probes_with_symbol:,}")
print(f"Probes without gene symbol: {probes_without_symbol:,}")
print(f"Saved local annotated ranking: {ANNOTATED_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

Probes in ranking: 49,386
Probes with gene symbol: 48,784
Probes without gene symbol: 602
Saved local annotated ranking: data\processed\tumor_vs_paired_normal_paired_probe_ranking_annotated.csv


## Top Annotated Rows

The rows below remain probe-level statistical results. Gene labels provide lookup context but do not convert the ranking into gene-level evidence.

In [6]:
top_annotated = annotated_ranking.loc[
    annotated_ranking["gene_symbol"].notna(),
    [
        "probe_id",
        "gene_symbol",
        "gene_title",
        "entrez_id",
        "mean_paired_difference",
        "adjusted_p_value",
    ],
].head(15)
display(top_annotated)

,probe_id,gene_symbol,gene_title,entrez_id,mean_paired_difference,adjusted_p_value
0,11728232_a_at,CLDN1,claudin 1,9076,4.920208,5.145358e-59
1,11735833_a_at,KIAA1199,KIAA1199,57214,4.699381,2.508659e-58
2,11719434_a_at,ETV4,ets variant 4,2118,3.297047,8.376406e-56
3,11728234_a_at,CLDN1,claudin 1,9076,4.224330,8.495091e-56
4,11721993_at,SLC6A6,solute carrier family 6 (neurotransmitter tran...,6533,3.303735,8.495091e-56
5,11733581_a_at,CA7,carbonic anhydrase VII,766,-4.651000,5.584336e-55
6,11739128_a_at,CDH3,"cadherin 3, type 1, P-cadherin (placental)",1001,3.592385,6.954774e-55
7,11759464_at,OTOP2,otopetrin 2,92736,-5.157509,5.053490e-54
8,11732838_at,GUCA2B,guanylate cyclase activator 2B (uroguanylin),2981,-6.547316,2.533048e-53
9,11721557_a_at,ABCA8,"ATP-binding cassette, sub-family A (ABC1), mem...",10351,-4.280251,2.923551e-52


## Examples of Many Probes Mapping to One Gene

Some annotation cells contain multiple symbols separated by `///`. They are split only for this mapping-count summary; the saved ranking retains the original platform annotation text.

In [7]:
symbol_probe_map = annotated_ranking.dropna(subset=["gene_symbol"])[
    ["probe_id", "gene_symbol"]
].copy()
symbol_probe_map["gene_symbol"] = symbol_probe_map["gene_symbol"].str.split(" /// ")
symbol_probe_map = symbol_probe_map.explode("gene_symbol")

multiple_probe_counts = (
    symbol_probe_map.groupby("gene_symbol")["probe_id"]
    .nunique()
    .sort_values(ascending=False)
)
multiple_probe_counts = multiple_probe_counts[multiple_probe_counts > 1]

mapping_examples = pd.DataFrame(
    {
        "probe_count": multiple_probe_counts.head(10),
        "example_probe_ids": [
            ", ".join(
                symbol_probe_map.loc[
                    symbol_probe_map["gene_symbol"] == symbol,
                    "probe_id",
                ].head(6)
            )
            for symbol in multiple_probe_counts.head(10).index
        ],
    }
)
display(mapping_examples)

,probe_count,example_probe_ids
gene_symbol,,
NF1,21,"11741948_a_at, 11731351_at, 11763466_at, 11751..."
RHD,19,"11736538_s_at, 11741979_at, 11740680_s_at, 117..."
TRD@,19,"11763447_x_at, 11761790_x_at, 11763446_s_at, 1..."
IGHM,18,"11759816_x_at, 11764046_x_at, 11759815_a_at, 1..."
FOXP1,17,"11758236_s_at, 11758158_s_at, 11750424_s_at, 1..."
IGKC,16,"11763253_x_at, 11763684_x_at, 11759652_x_at, 1..."
FMNL1,16,"11763558_a_at, 11763644_x_at, 11763689_x_at, 1..."
DMKN,16,"11749409_a_at, 11759755_x_at, 11741539_x_at, 1..."
NFATC4,16,"11734672_a_at, 11761235_x_at, 11754045_x_at, 1..."


## Annotation Limitations

- Probe-to-gene mapping is many-to-one for many symbols and may also be one-to-many within individual annotation fields.
- Some probes lack a gene symbol even when another annotation field is present.
- The GPL13667 table records an older annotation date and GRCh37-era coordinates, so identifiers may differ from current resources.
- Multiple probes for one symbol are not independent gene-level confirmations and should not be collapsed without a documented rule.
- Annotation enables careful lookup, but biological interpretation still requires validation of preprocessing, probe behavior, current identifiers, and the paired analysis assumptions.